# DLC 통합 프로젝트 파이프라인

| 단계 | 내용 | 실행 시점 |
|------|------|-----------|
| STEP 0 | 통합 프로젝트 생성 | **처음 한 번만** |
| STEP 1 | 기존 라벨링 폴더 흡수 | **처음 한 번만** (추가 시 재실행) |
| STEP 2 | 학습 데이터셋 생성 + 모델 학습 | **처음 한 번만** (재학습 시 재실행) |
| STEP 3 | 새 비디오 분석 | **새 비디오 생길 때마다** |

> ✅ 새 비디오(10048, 10049...)가 생길 때는 **STEP 3만** 실행하면 됩니다.


### 처음 사용 시 한 번만 실행

In [ ]:
pip install deeplabcut imgaug napari

## 라이브러리 임포트

In [1]:
import deeplabcut
import os
import shutil
import yaml
import tkinter as tk
from tkinter import filedialog, simpledialog

print("✅ 라이브러리 로드 완료")

Loading DLC 3.0.0rc13...


c:\Users\USER\miniconda3\envs\dlc-new\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 라이브러리 로드 완료


## GUI 헬퍼 함수 (수정 불필요)

In [2]:
def _root():
    r = tk.Tk()
    r.withdraw()
    r.attributes("-topmost", True)
    return r

def ask_string(title, prompt, initial=""):
    r = _root()
    val = simpledialog.askstring(title, prompt, initialvalue=initial, parent=r)
    r.destroy()
    return val

def pick_file(title, filetypes=None):
    r = _root()
    path = filedialog.askopenfilename(
        title=title,
        filetypes=filetypes or [("All files", "*.*")]
    )
    r.destroy()
    return path

def pick_directory(title):
    r = _root()
    path = filedialog.askdirectory(title=title)
    r.destroy()
    return path

def pick_folders_loop(prompt_prefix):
    """폴더를 취소할 때까지 반복 선택 (개수 제한 없음)"""    
    selected = []
    print(f"  폴더를 하나씩 선택하세요. 더 이상 없으면 '취소'를 누르세요.")
    while True:
        r = _root()
        folder = filedialog.askdirectory(
            title=f"{prompt_prefix} ({len(selected)+1}번째) — 끝나면 '취소'"
        )
        r.destroy()
        if not folder:
            break
        if folder in selected:
            print(f"  ⚠️  중복: {os.path.basename(folder)}")
        else:
            selected.append(folder)
            print(f"  ✅ 추가: {os.path.basename(folder)}  ({len(selected)}개)")
    return selected

def pick_videos_loop():
    """비디오를 취소할 때까지 반복 선택 (개수 제한 없음)"""    
    selected = []
    print("  비디오를 하나씩 선택하세요. 더 이상 없으면 '취소'를 누르세요.")
    while True:
        r = _root()
        f = filedialog.askopenfilename(
            title=f"비디오 선택 ({len(selected)+1}번째) — 끝나면 '취소'",
            filetypes=[("Video files", "*.mp4 *.avi *.mov *.mkv"), ("All files", "*.*")]
        )
        r.destroy()
        if not f:
            break
        if f in selected:
            print(f"  ⚠️  중복: {os.path.basename(f)}")
        else:
            selected.append(f)
            print(f"  ✅ 추가: {os.path.basename(f)}  ({len(selected)}개)")
    return selected

print("✅ 헬퍼 함수 로드 완료")

✅ 헬퍼 함수 로드 완료


---
## STEP 0 — 통합 프로젝트 생성
> ⚠️ **처음 한 번만 실행하세요.**  
> 이미 통합 프로젝트가 있으면 이 셀을 건너뛰고 바로 `config_path`를 아래에서 직접 입력하세요.

In [3]:
# ── 프로젝트 이름 / 연구자 이름 입력 ──────────────────────
proj_name  = ask_string("프로젝트 이름", "통합 프로젝트 이름", "mouse_behavior")
researcher = ask_string("연구자 이름",   "연구자 이름",       "YourName")

# ── 프로젝트 생성 위치 선택 ────────────────────────────────
print("프로젝트를 생성할 폴더를 선택하세요.")
working_dir = pick_directory("프로젝트 생성 위치")

# ── 대표 비디오 1개 선택 (프로젝트 생성에 필요) ────────────
print("대표 비디오를 1개 선택하세요.")
seed_video = pick_file(
    "대표 비디오 선택",
    [("Video files", "*.mp4 *.avi *.mov *.mkv"), ("All files", "*.*")]
)

# ── 프로젝트 생성 ──────────────────────────────────────────
config_path = deeplabcut.create_new_project(
    proj_name,
    researcher,
    [seed_video],
    working_directory=working_dir,
    copy_videos=False
)
print(f"\n✅ 프로젝트 생성 완료: {config_path}")

# ── config.yaml 자동 설정 ──────────────────────────────────
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

# Body parts 설정 (필요에 따라 수정)
cfg['bodyparts'] = [
    'nose',
    'R_ear',
    'L_ear',
    'neck',
    'body_center',
    'tail_base',
    'tail_end'
]

# 학습/테스트 비율 설정 (0.85 = 학습 85%, 테스트 15%)
cfg['TrainingFraction'] = [0.85]

# Skeleton 설정
cfg['skeleton'] = [
    ['nose',        'R_ear'      ],
    ['nose',        'L_ear'      ],
    ['nose',        'neck'       ],
    ['neck',        'body_center'],
    ['body_center', 'tail_base'  ],
    ['tail_base',   'tail_end'   ]
]

# 프레임 추출 수 (영상 길이에 따라 조정)
cfg['numframes2pick'] = 50

# 시각화 설정
cfg['dotsize']        = 2
cfg['colormap']       = 'rainbow'
cfg['skeleton_color'] = 'white'

with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print("\n✅ config.yaml 자동 설정 완료")
print(f"   bodyparts      : {cfg['bodyparts']}")
print(f"   numframes2pick : {cfg['numframes2pick']}")
print(f"   dotsize        : {cfg['dotsize']}")
print(f"   colormap       : {cfg['colormap']}")
print(f"   skeleton_color : {cfg['skeleton_color']}")
print("\n💡 bodyparts / skeleton 을 바꾸려면 이 셀을 수정 후 재실행하세요.")

프로젝트를 생성할 폴더를 선택하세요.
대표 비디오를 1개 선택하세요.
Created "E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\videos"
Created "E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\labeled-data"
Created "E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\training-datasets"
Created "E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\dlc-models"
Attempting to create a symbolic link of the video ...
Symlink creation impossible (exFat architecture?): copying the video instead.
E:\OSH\Day0\10281_2026-07-16_08-21-43\behavior-videos\10281_topcamera_2026-07-16T08_11_31.avi copied to E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\videos\10281_topcamera_2026-07-16T08_11_31.avi
E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\videos\10281_topcamera_2026-07-16T08_11_31.avi
Generated "E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\config.yaml"

A new project with name Thirst_Threat_RJH-OSH-2026-07-20 is created at E:\OSH and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt to your project's needs.
 Once you have changed the configur

---
## (STEP 0 건너뛴 경우) 기존 config.yaml 경로 설정
> STEP 0을 실행했다면 이 셀은 건너뛰세요.

In [3]:
# STEP 0을 건너뛴 경우: config.yaml 을 GUI로 선택
print("기존 통합 프로젝트의 config.yaml 을 선택하세요.")
config_path = pick_file(
    "config.yaml 선택",
    [("YAML files", "*.yaml *.yml"), ("All files", "*.*")]
)
print(f"✅ config_path = {config_path}")

기존 통합 프로젝트의 config.yaml 을 선택하세요.
✅ config_path = E:/OSH/Thirst_Threat_RJH-OSH-2026-07-20/config.yaml


---
## STEP 1 — 기존 라벨링 폴더 흡수
> ⚠️ **처음 한 번만 실행하세요.** (새 라벨링 폴더를 추가할 때는 재실행)  
> 10040, 10028, 10042 등 라벨링이 완료된 폴더를 선택합니다.  
> 개수 제한 없음 — 취소를 누를 때까지 계속 추가 가능합니다.

* .h5 파일 연구자 이름을 프로젝트 연구자 이름과 통일 
   ex) CollectedData_Choi -> CollectedData_YB

* 저장경로 확인

In [4]:
import pandas as pd

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)
project_path   = cfg["project_path"]
project_scorer = cfg["scorer"]
labeled_dir    = os.path.join(project_path, "labeled-data")
os.makedirs(labeled_dir, exist_ok=True)

# ── 라벨링 폴더 선택 ──
src_folders = pick_folders_loop("라벨링 폴더 선택")

print(f"\n[ 검증 및 복사 ]  (project scorer: {project_scorer})")
for src in src_folders:
    name    = os.path.basename(src)
    dst     = os.path.join(labeled_dir, name)
    h5_list = [f for f in os.listdir(src)
               if f.startswith("CollectedData_") and f.endswith(".h5")]

    if not h5_list:
        print(f"  ❌ {name}: CollectedData_*.h5 없음 → 건너뜀")
        continue

    if os.path.abspath(src) == os.path.abspath(dst):
        print(f"  ✔  {name}: 이미 프로젝트 내부")
    elif os.path.exists(dst):
        print(f"  ✔  {name}: 이미 존재 (복사 생략)")
    else:
        shutil.copytree(src, dst)
        print(f"  📂 {name}: 복사 완료")

    # ── scorer 이름 정규화 ──
    for h5_file in [f for f in os.listdir(dst)
                    if f.startswith("CollectedData_") and f.endswith(".h5")]:
        src_scorer = h5_file[len("CollectedData_"):-len(".h5")]
        if src_scorer == project_scorer:
            print(f"     scorer 일치 ({project_scorer}) ✔")
            continue

        old_h5  = os.path.join(dst, h5_file)
        new_h5  = os.path.join(dst, f"CollectedData_{project_scorer}.h5")
        old_csv = os.path.join(dst, f"CollectedData_{src_scorer}.csv")
        new_csv = os.path.join(dst, f"CollectedData_{project_scorer}.csv")

        df = pd.read_hdf(old_h5)
        df.columns = pd.MultiIndex.from_tuples(
            [(project_scorer if lvl0 == src_scorer else lvl0, *rest)
             for lvl0, *rest in df.columns],
            names=df.columns.names
        )
        df.to_hdf(new_h5, key="df_with_missing", mode="w")
        os.remove(old_h5)

        if os.path.exists(old_csv):
            with open(old_csv, "r", encoding="utf-8") as f:
                text = f.read()
            with open(new_csv, "w", encoding="utf-8") as f:
                f.write(text.replace(src_scorer, project_scorer, 1))
            os.remove(old_csv)

        print(f"     scorer 변환: {src_scorer} → {project_scorer}  ✔")

# ── 대응 영상 선택 → video_sets 등록 ──
print("\n[ 영상 등록 ]")
print("  각 동물의 원본 영상을 선택하세요. (없으면 '취소'로 건너뜀)")
existing_videos = set(cfg.get("video_sets", {}).keys())
new_videos = []
for src in src_folders:
    name = os.path.basename(src)
    # 이미 등록된 이름이 있으면 건너뜀
    if any(name.replace("_", "") in v.replace("_", "") for v in existing_videos):
        print(f"  ✔  {name}: 이미 video_sets 등록됨")
        continue
    r = tk.Tk(); r.withdraw(); r.attributes("-topmost", True)
    vid = filedialog.askopenfilename(
        title=f"{name} 영상 파일 선택 (없으면 취소)",
        filetypes=[("Video", "*.avi *.mp4 *.mov *.mkv"), ("All", "*.*")]
    )
    r.destroy()
    if vid:
        new_videos.append(vid)
        print(f"  ✅ {name}: {os.path.basename(vid)}")
    else:
        print(f"  ⏭  {name}: 건너뜀")

if new_videos:
    deeplabcut.add_new_videos(config_path, new_videos, copy_videos=False)
    print(f"\n  → {len(new_videos)}개 영상 video_sets 등록 완료")

print("\n✅ STEP 1 완료")

  폴더를 하나씩 선택하세요. 더 이상 없으면 '취소'를 누르세요.
  ✅ 추가: 10281_topcamera_2026-07-16T08_11_31  (1개)
  ✅ 추가: 10284_topcamera_2026-07-16T10_47_40  (2개)
  ✅ 추가: 10285_topcamera_2026-07-16T09_57_03  (3개)
  ✅ 추가: 10281_topcamera_2026-07-19T08_09_54  (4개)
  ✅ 추가: 10282_topcamera_2026-07-19T09_58_08  (5개)
  ✅ 추가: 10283_topcamera_2026-07-19T08_54_33  (6개)

[ 검증 및 복사 ]  (project scorer: OSH)
  ✔  10281_topcamera_2026-07-16T08_11_31: 이미 존재 (복사 생략)
  📂 10284_topcamera_2026-07-16T10_47_40: 복사 완료
     scorer 일치 (OSH) ✔
  📂 10285_topcamera_2026-07-16T09_57_03: 복사 완료
     scorer 일치 (OSH) ✔
  📂 10281_topcamera_2026-07-19T08_09_54: 복사 완료
     scorer 일치 (OSH) ✔
  📂 10282_topcamera_2026-07-19T09_58_08: 복사 완료
     scorer 일치 (OSH) ✔
  📂 10283_topcamera_2026-07-19T08_54_33: 복사 완료
     scorer 일치 (OSH) ✔

[ 영상 등록 ]
  각 동물의 원본 영상을 선택하세요. (없으면 '취소'로 건너뜀)
  ✔  10281_topcamera_2026-07-16T08_11_31: 이미 video_sets 등록됨
  ✅ 10284_topcamera_2026-07-16T10_47_40: 10283_topcamera_2026-07-19T08_54_33.avi
  ✅ 10285_topcamera_2026-07-16

---
## STEP 2-1 — 학습 데이터셋 생성
> ⚠️ **처음 한 번만 실행하세요.** (데이터 추가 후 재학습 시에는 재실행)

프로젝트 파일 dlc-models-pytorch에서 최신 Shuffles을 1로 수정
-> 그전 shuffles 파일은 삭제

In [5]:
deeplabcut.create_training_dataset(
    config_path,
    num_shuffles=1,
    augmenter_type="imgaug"
)
print("✅ 학습 데이터셋 생성 완료")

E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\labeled-data\10281_topcamera_2026-07-16T08_11_31\CollectedData_OSH.h5  not found (perhaps not annotated).
✅ 학습 데이터셋 생성 완료


## STEP 2-2 — 모델 학습
> ⚠️ **처음 한 번만 실행하세요.**  
> `maxiters`: 데이터가 적으면 15000도 충분합니다.  
> `gputouse`: CPU만 있으면 해당 줄을 삭제하세요.

In [9]:
deeplabcut.train_network(
    config_path,
    shuffle=1,
    trainingsetindex=0,
    maxiters=50000,
    displayiters=1000,
    saveiters=5000,
    batch_size=16,
    gputouse=0          # CPU만 있으면 이 줄 삭제
)
print("✅ 모델 학습 완료")

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
inference:
  multithreading:
    enabled: True
    queue_length: 4
    timeout: 30.0
  compile:
    enabled: False
    backend: inductor
  autocast:
    enabled: False
metadata:
  project_path: E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20
  pose_config_path: E:\OSH\Thirst_Threat_RJH-OSH-2026-07-20\dlc-models-pytorch\iteration-0\Thirst_Threat_RJHJul20-trainset85shuffle1\train\pytorch_config.yaml
  bodyparts: ['nose', 'R_ear', 'L_ear', 'neck', 'body_center', 'tail_base', 'tail_end']
  unique_bodyparts: []
  individuals: ['animal']
  with_identity: None
method: bu
model:
  backbone:
    type: ResNet
    mo

✅ 모델 학습 완료


## STEP 2-3 — 모델 평가

In [10]:
deeplabcut.evaluate_network(
    config_path,
    Shuffles=[1],
    plotting=True
)
print("✅ 모델 평가 완료")

Evaluation scorer: DLC_Resnet50_Thirst_Threat_RJHJul20shuffle1_snapshot_best-160


100%|██████████| 41/41 [00:01<00:00, 21.49it/s]


Evaluation results file: DLC_Resnet50_Thirst_Threat_RJHJul20shuffle1_snapshot_best-160-results.csv
Evaluation results for DLC_Resnet50_Thirst_Threat_RJHJul20shuffle1_snapshot_best-160-results.csv (pcutoff: 0.6):
train rmse             1.29
train rmse_pcutoff     1.26
train mAP             96.98
train mAR             97.70
test rmse              1.97
test rmse_pcutoff      1.93
test mAP              89.09
test mAR              91.95
Name: (0.85, 1, 160, -1, 0.6), dtype: float64
✅ 모델 평가 완료


---
## STEP 3 — 새 비디오 분석 ⬅ 반복 사용
> ✅ **새 비디오가 생길 때마다 이 셀부터 실행하세요.**  
> 비디오를 하나씩 선택 → 취소를 누르면 선택 완료.

In [4]:
videos = pick_videos_loop()
print(f"\n→ {len(videos)}개 비디오 분석 시작")

  비디오를 하나씩 선택하세요. 더 이상 없으면 '취소'를 누르세요.

→ 0개 비디오 분석 시작


### 3-1. 비디오 분석

In [ ]:
import os, glob, yaml, torch

shuffle = 1
torch.backends.cudnn.benchmark = True   # ① conv 알고리즘 autotune (추론 속도 향상)

# ② autocast(FP16) 자동 켜기 — 재학습(create_training_dataset)으로 설정이 리셋돼도 매 실행 시 다시 켜줌
with open(config_path) as f:
    _cfg = yaml.safe_load(f)
_base = os.path.join(_cfg["project_path"], "dlc-models-pytorch")
_it = _cfg.get("iteration", 0)
_pcfgs = (glob.glob(os.path.join(_base, f"iteration-{_it}", f"*shuffle{shuffle}", "train", "pytorch_config.yaml"))
          or sorted(glob.glob(os.path.join(_base, "iteration-*", f"*shuffle{shuffle}", "train", "pytorch_config.yaml"))))
if _pcfgs:
    _p = _pcfgs[-1]
    with open(_p) as f:
        _pc = yaml.safe_load(f)
    _pc.setdefault("inference", {}).setdefault("autocast", {})
    if _pc["inference"]["autocast"].get("enabled") is not True:
        _pc["inference"]["autocast"]["enabled"] = True
        with open(_p, "w") as f:
            yaml.safe_dump(_pc, f, sort_keys=False, allow_unicode=True)
        print("⚡ autocast(FP16) 활성화함")
    else:
        print("⚡ autocast(FP16) 이미 켜져 있음")
else:
    print("⚠️ pytorch_config.yaml 못 찾음 — autocast 수동 확인 필요")

# ③ 분석 (이미 분석된 영상은 자동 skip)
deeplabcut.analyze_videos(
    config_path,
    videos,
    shuffle=shuffle,
    save_as_csv=True,
    batchsize=16
)
print("✅ 분석 완료 → .h5 / .csv 생성됨")

### 3-2. 트래킹 필터 적용 (튐 현상 제거)

In [ ]:
for video in videos:
    print(f"  필터 적용 중: {os.path.basename(video)}")
    deeplabcut.filterpredictions(
        config_path,
        [video],
        shuffle=1,
        filtertype="median",
        windowlength=5
    )
print("✅ 필터 완료 → _filtered.h5 생성됨")

### 3-3. 라벨 오버레이 비디오 생성

In [ ]:
deeplabcut.create_labeled_video(
    config_path,
    videos,
    shuffle=1,
    filtered=True,
    draw_skeleton=True,
    dotsize=6,
    colormap="rainbow"
)
print("✅ 라벨 비디오 생성 완료")

## 4. 아웃라이어 추출 & 보정 (선택)
분석 결과에서 튀는 프레임을 자동 추출하고 재라벨링 후 재학습합니다.

In [ ]:
# Step 1: 아웃라이어 프레임 추출
deeplabcut.extract_outlier_frames(
    config_path,
    videos=videos,
    outlieralgorithm='jump',  # 'jump': 급격한 위치 변화 감지
    epsilon=20,               # 픽셀 오차 임계값 (필요시 조정)
    p_bound=0.01,
    automatic=True,
)

print("아웃라이어 프레임 추출 완료!")

In [ ]:
# Step 2: 아웃라이어 프레임 재라벨링 (napari 실행)
deeplabcut.refine_labels(config_path)

In [ ]:
# Step 3: 데이터셋 병합
deeplabcut.merge_datasets(config_path)

print("데이터셋 병합 완료!")

In [ ]:
# Step 4: 재학습 (이전보다 적은 iter)
deeplabcut.create_training_dataset(config_path)

deeplabcut.train_network(
    config_path,
    shuffle=1,
    displayiters=500,
    saveiters=5000,
    maxiters=15000,
    batch_size=16,       
)

print("재학습 완료!")